# Re-judge v0 with the 8B judge

The 4B model's answers are saved in `data/runs/answers-v0-dev.jsonl` (with the 4B self-judge verdict in `correct`).
Here **Qwen3-VL-8B-Instruct-FP8** grades the same answers with the **same prompt**, so the only change is the judge.

**Before running:** stop the 4B server and serve the 8B model instead (see the reply in chat).

In [1]:
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from openai import OpenAI
from tqdm.auto import tqdm

BASE_URL = "http://127.0.0.1:8000/v1"
JUDGE_MODEL = "Qwen/Qwen3-VL-8B-Instruct-FP8"
WORKERS = 8

DATA = Path("../data/runs") if Path("../data").is_dir() else Path("data/runs")
rows = [json.loads(line) for line in (DATA / "answers-v0-dev.jsonl").read_text().splitlines()]

client = OpenAI(base_url=BASE_URL, api_key="none")
print(len(rows), "saved answers | server serves:", [m.id for m in client.models.list()])

/storage/homefs/kr23w045/vdoc-agent/envs/train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


108 saved answers | server serves: ['Qwen/Qwen3-VL-8B-Instruct-FP8']


Same judge prompt and schema as notebook 04.

In [2]:
JUDGE = (
    "You grade an answer against a reference answer. The answer is correct if it states the same key facts as the "
    "reference: numbers, names and dates must match. Extra detail is fine. Missing or wrong key facts make it incorrect. "
    "An empty answer is incorrect."
)
JUDGE_SCHEMA = {"type": "object", "properties": {"correct": {"type": "boolean"}}, "required": ["correct"]}

def judge(r):
    prompt = f"Question: {r['query']}\nReference answer: {r['answer']}\nAnswer to grade: {r['pred']['answer']}"
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "system", "content": JUDGE}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema", "json_schema": {"name": "grade", "schema": JUDGE_SCHEMA}},
        temperature=0,
        max_tokens=20,
    )
    return json.loads(resp.choices[0].message.content)["correct"]

with ThreadPoolExecutor(WORKERS) as pool:
    verdicts_8b = list(tqdm(pool.map(judge, rows), total=len(rows)))

100%|██████████| 108/108 [00:05<00:00, 19.77it/s]


## Accuracy: 4B self-judge vs 8B judge

In [3]:
def share(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else float("nan")

for r, v in zip(rows, verdicts_8b):
    r["correct_8b"] = v

for name, key in [("4B self-judge", "correct"), ("8B judge", "correct_8b")]:
    print(f"{name:14} accuracy {share(r[key] for r in rows):.3f} | "
          f"evidence shown {share(r[key] for r in rows if r['evidence_shown']):.3f} | "
          f"not shown {share(r[key] for r in rows if not r['evidence_shown']):.3f}")

4B self-judge  accuracy 0.500 | evidence shown 0.588 | not shown 0.174
8B judge       accuracy 0.435 | evidence shown 0.518 | not shown 0.130


## Where do the two judges disagree?

`(4B verdict, 8B verdict)` counts. `(True, False)` = the 4B judge passed an answer the 8B judge fails: the self-judge being
lenient. Read a few disagreements and decide yourself which judge is right; that is the start of P12's human labels.

In [4]:
print(Counter((r["correct"], r["correct_8b"]) for r in rows))
print(f"agreement: {share(r['correct'] == r['correct_8b'] for r in rows):.3f}\n")

for r in [r for r in rows if r["correct"] != r["correct_8b"]][:5]:
    print("Q:   ", r["query"])
    print("REF: ", r["answer"])
    print("PRED:", r["pred"]["answer"])
    print("4B says", r["correct"], "| 8B says", r["correct_8b"], "\n")

Counter({(False, False): 53, (True, True): 46, (True, False): 8, (False, True): 1})
agreement: 0.917

Q:    Examine the role of visual presentation in shaping consumer beliefs about drug effectiveness based on the described research initiatives.
REF:  Visual presentation shapes consumer beliefs about drug effectiveness by influencing attention, comprehension, and interpretation of drug information. Factors such as ad similarity, placement, design elements, disclosure prominence, and context affect how consumers perceive the benefits and risks of a drug, demonstrating that layout and formatting can alter their understanding.
PRED: The research initiatives described, such as the Experimental Study of an Accelerated Approval Disclosure (Page 2136) and the Role of Research Team (Page 2146), highlight the importance of visual presentation in shaping consumer beliefs about drug effectiveness. The study on accelerated approval disclosure specifically varies the presence, prominence, and reada

In [5]:
from vdoc_agent.runlog import log_run

metrics = {
    "accuracy": share(r["correct_8b"] for r in rows),
    "accuracy_evidence_shown": share(r["correct_8b"] for r in rows if r["evidence_shown"]),
    "accuracy_evidence_not_shown": share(r["correct_8b"] for r in rows if not r["evidence_shown"]),
    "agreement_with_4b_judge": share(r["correct"] == r["correct_8b"] for r in rows),
}
config = {"answers": "data/runs/answers-v0-dev.jsonl", "answer_model": "Qwen/Qwen3-VL-4B-Instruct",
          "judge": JUDGE_MODEL, "split": "dev", "temperature": 0}
print(log_run("judge-8b-v0", config, metrics))

/storage/homefs/kr23w045/vdoc-agent/reports/runs/20261007T020224Z_judge-8b-v0.json
